# Detailed table-of-contents exploration

Explore the saved 9,034-record catalog before choosing preprocessing or chunking rules. Each raw row is an Amazon parent record, and several rows can share an Open Library edition. Counts describe this snapshot only.

Run all cells from the project's `.venv` kernel. This notebook reads `data/raw/books.parquet` and creates inspection tables in memory. Start with the source comparison, then inspect coverage, extraction, structure, and text quality.

## 1. Load the local snapshot

Objective: Load the saved books and record the file checksum before exploration.

In [1]:
import hashlib
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from bookpath.local_data import read_books_from_parquet

project_root = Path.cwd()
while not (project_root / "src/bookpath").is_dir():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the Bookpath repository.")
    project_root = project_root.parent

raw_path = project_root / "data/raw/books.parquet"


def file_checksum(path):
    with path.open("rb") as raw_file:
        return hashlib.file_digest(raw_file, "sha256").hexdigest()


checksum_before = file_checksum(raw_path)
books = read_books_from_parquet(raw_path)
print(f"Books: {len(books):,}; checksum: {checksum_before}")

Books: 9,034; checksum: 4acf6979b15215c28f2a45a1488ace4d749e2573f0fa4a0110e2933dd0194dd8


## 2. Locate the original TOC

Objective: Check that our separate `raw_toc_json` column contains the same original TOC as the one inside `openlibrary_raw_edition_json`. We decode both JSON strings first, then compare their Python lists item by item, including order.

For example, one cell stores a JSON string like `'[{"title": "1. Nearly meeting"}]'`. After `json.loads`, it becomes a Python list containing a dictionary: `[{"title": "1. Nearly meeting"}]`. The full Open Library JSON becomes a dictionary, from which we select its `table_of_contents` list.

This first check compares **two copies of the original TOC**. We compare original items with the already-parsed `toc_entries` later. Decoding JSON changes how we access the data; it does not clean the text.

In [2]:
# Each cell starts as a JSON string; decode it into a Python list of TOC items.
raw_tocs = books["raw_toc_json"].map(json.loads)

# Each full Open Library JSON string becomes a Python dictionary.
openlibrary_records = books["openlibrary_raw_edition_json"].map(json.loads)
# Select the original TOC list from each dictionary.
openlibrary_tocs = openlibrary_records.map(lambda record: record.get("table_of_contents"))

# Pair the two lists for the same book. Equality checks their contents and order.
# strict=True also checks that we compare the same number of books on both sides.
source_matches = pd.Series(
    [raw == original for raw, original in zip(raw_tocs, openlibrary_tocs, strict=True)],
    index=books.index,
)
print(f"Identical raw and Open Library TOCs: {source_matches.sum():,} / {len(books):,}")
display(books["raw_toc_shape"].value_counts().rename("books").to_frame())

Identical raw and Open Library TOCs: 9,034 / 9,034


,books
raw_toc_shape,
list[dict],9030
list[string],4


Objective: Display one book's original TOC items and its parsed entries as two tables. Change `book_position` to inspect another record; positions start at zero. Original item positions and parsed sequence numbers can differ when an empty item was omitted.

`selected_book = books.iloc[book_position]` selects the **whole row**, as a Pandas `Series`. `selected_book["title"]` gets the book-title string from that row. For the first parsed TOC entry's text, use `selected_book["toc_entries"][0]["text"]`. Selecting these values does not clean them.

In this Parquet snapshot, `toc_entries` is a NumPy array of dictionaries. `list()` converts that array to a Python list while keeping the same dictionaries in order. Passing this list to `DataFrame` gives one column per dictionary key (`text`, `page`, etc.). Passing the array directly gives a single column containing whole dictionaries.

In [12]:
book_position = 3
# Select a whole book row (a Pandas Series), then its original TOC (a Python list).
selected_book = books.iloc[book_position]
selected_raw_toc = raw_tocs.iloc[book_position]
raw_view = pd.DataFrame({
    "raw_position": range(1, len(selected_raw_toc) + 1),
    "raw_item": selected_raw_toc,
})

# NumPy array of dictionaries -> Python list of dictionaries -> table of fields.
# list() changes the container for display; it does not clean any entry text.
parsed_entries = list(selected_book["toc_entries"])
parsed_view = pd.DataFrame(parsed_entries)

print(f"Row position: {book_position}; parent_asin: {selected_book['parent_asin']}")
print(f"Amazon title: {selected_book['title']}")
print(f"Open Library title: {selected_book['ol_title']}")
with pd.option_context("display.max_colwidth", None):
    print("Original TOC: each raw_item cell contains an entire original item.")
    display(raw_view)
    print("Parsed TOC: each dictionary field has its own column.")
    display(parsed_view)

Row position: 3; parent_asin: 0026441918
Amazon title: Marketing Essentials, Third Edition
Open Library title: Marketing essentials


,raw_position,raw_item
0,1,"{'title': 'The world of marketing', 'type': {'key': '/type/toc_item'}, 'level': 0}"
1,2,"{'title': 'Economics', 'type': {'key': '/type/toc_item'}, 'level': 0}"
2,3,"{'title': 'Business and international marketing', 'type': {'key': '/type/toc_item'}, 'level': 0}"
3,4,"{'title': 'Academic concepts and skills', 'type': {'key': '/type/toc_item'}, 'level': 0}"
4,5,"{'title': 'Selling', 'type': {'key': '/type/toc_item'}, 'level': 0}"
5,6,"{'title': 'Promotion', 'type': {'key': '/type/toc_item'}, 'level': 0}"
6,7,"{'title': 'Distribution', 'type': {'key': '/type/toc_item'}, 'level': 0}"
7,8,"{'title': 'Pricing', 'type': {'key': '/type/toc_item'}, 'level': 0}"
8,9,"{'title': 'Marketing information management', 'type': {'key': '/type/toc_item'}, 'level': 0}"
9,10,"{'title': 'Product and service management', 'type': {'key': '/type/toc_item'}, 'level': 0}"


,0
0,"{'level': '0', 'page': None, 'sequence': '1', 'source_key': 'title', 'text': 'The world of marketing'}"
1,"{'level': '0', 'page': None, 'sequence': '2', 'source_key': 'title', 'text': 'Economics'}"
2,"{'level': '0', 'page': None, 'sequence': '3', 'source_key': 'title', 'text': 'Business and international marketing'}"
3,"{'level': '0', 'page': None, 'sequence': '4', 'source_key': 'title', 'text': 'Academic concepts and skills'}"
4,"{'level': '0', 'page': None, 'sequence': '5', 'source_key': 'title', 'text': 'Selling'}"
5,"{'level': '0', 'page': None, 'sequence': '6', 'source_key': 'title', 'text': 'Promotion'}"
6,"{'level': '0', 'page': None, 'sequence': '7', 'source_key': 'title', 'text': 'Distribution'}"
7,"{'level': '0', 'page': None, 'sequence': '8', 'source_key': 'title', 'text': 'Pricing'}"
8,"{'level': '0', 'page': None, 'sequence': '9', 'source_key': 'title', 'text': 'Marketing information management'}"
9,"{'level': '0', 'page': None, 'sequence': '10', 'source_key': 'title', 'text': 'Product and service management'}"


## 3. Inspect every raw item

Objective: Build an inspection table with the book ID, original item position, and candidate text. We test the observed fallback `title` → `value` → `label`; this is a comparison hypothesis, not a new production parser.

In [ ]:
def candidate_text_and_key(item):
    if isinstance(item, str):
        return item, None
    for key in ["title", "value", "label"]:
        value = item.get(key)
        if isinstance(value, str) and value.strip():
            return value, key
    return None, None


raw_rows = []
for parent_asin, raw_toc in zip(books["parent_asin"], raw_tocs, strict=True):
    for raw_position, item in enumerate(raw_toc, start=1):
        text, source_key = candidate_text_and_key(item)
        raw_rows.append({
            "parent_asin": parent_asin,
            "raw_position": raw_position,
            "raw_item": item,
            "item_type": type(item).__name__,
            "candidate_text": text,
            "candidate_source_key": source_key,
        })

raw_items = pd.DataFrame(raw_rows)
print(f"Raw items: {len(raw_items):,}")
display(raw_items["item_type"].value_counts().rename("items").to_frame())

Objective: Count every property found inside raw TOC dictionaries. These properties may contain information beyond chapter titles. Counts are per raw item and per Amazon parent record.

In [ ]:
raw_property_rows = []
for row in raw_items.itertuples():
    if isinstance(row.raw_item, dict):
        for key, value in row.raw_item.items():
            raw_property_rows.append({
                "parent_asin": row.parent_asin,
                "raw_position": row.raw_position,
                "property": key,
                "value": value,
            })

raw_properties = pd.DataFrame(raw_property_rows)
property_summary = raw_properties.groupby("property").agg(
    items_with_key=("raw_position", "size"),
    books_with_key=("parent_asin", "nunique"),
)
display(property_summary.sort_index())

## 4. Explain count differences

Objective: Compare both stored counts with actual list lengths, and count raw items without usable title/value/label text. A count difference alone does not prove useful content was lost.

In [ ]:
candidate_text = raw_items["candidate_text"].astype("string")
raw_items["has_candidate_text"] = candidate_text.fillna("").str.strip().ne("")
unusable_items = raw_items.loc[~raw_items["has_candidate_text"]]

toc_counts = books[["parent_asin", "title", "raw_toc_shape"]].copy()
toc_counts["stored_raw_count"] = pd.to_numeric(books["raw_toc_item_count"])
toc_counts["actual_raw_count"] = raw_tocs.map(len)
toc_counts["stored_parsed_count"] = pd.to_numeric(books["toc_entry_count"])
toc_counts["actual_parsed_count"] = books["toc_entries"].map(len)
toc_counts["items_without_text"] = (
    toc_counts["parent_asin"].map(unusable_items.groupby("parent_asin").size()).fillna(0).astype(int)
)
toc_counts["raw_minus_parsed"] = toc_counts["actual_raw_count"] - toc_counts["actual_parsed_count"]
print("Stored raw count mismatches:", toc_counts["stored_raw_count"].ne(toc_counts["actual_raw_count"]).sum())
print("Stored parsed count mismatches:", toc_counts["stored_parsed_count"].ne(toc_counts["actual_parsed_count"]).sum())
print("Raw items without usable text:", len(unusable_items))
display(toc_counts.loc[toc_counts["raw_minus_parsed"].ne(0)])
display(unusable_items[["parent_asin", "raw_position", "raw_item"]])

## 5. Inspect parsed fields and preservation

Objective: Create one inspection row per parsed entry, retaining its parent book and actual position. This table is for analysis, not a training split or a saved processed dataset.

In [ ]:
parsed_rows = []
for parent_asin, entries in zip(books["parent_asin"], books["toc_entries"], strict=True):
    for parsed_position, entry in enumerate(entries, start=1):
        parsed_rows.append({
            **entry,
            "parent_asin": parent_asin,
            "parsed_position": parsed_position,
        })

parsed_items = pd.DataFrame(parsed_rows)
print(f"Parsed entries: {len(parsed_items):,}")
display(parsed_items["source_key"].value_counts(dropna=False).rename("entries").to_frame())

Objective: Compare each usable raw item's text, source key, level, and page with its corresponding parsed entry. Candidate positions are assigned after omitting raw items without usable text. All disagreements stay available in `preservation_review`.

In [ ]:
usable_raw_items = raw_items.loc[raw_items["has_candidate_text"]].copy()
usable_raw_items["parsed_position"] = usable_raw_items.groupby("parent_asin").cumcount() + 1
preservation_review = usable_raw_items.merge(
    parsed_items, on=["parent_asin", "parsed_position"], how="outer", indicator=True,
    validate="one_to_one",
)
preservation_review["text_matches"] = (
    preservation_review["candidate_text"].astype("string").str.strip()
    .eq(preservation_review["text"].astype("string")).fillna(False)
)
preservation_review["source_key_matches"] = (
    preservation_review["candidate_source_key"].fillna("<missing>")
    .eq(preservation_review["source_key"].fillna("<missing>"))
)

def raw_field_as_text(item, field):
    value = item.get(field) if isinstance(item, dict) else None
    return None if value is None else str(value)


for raw_field, parsed_field in [("level", "level"), ("pagenum", "page")]:
    raw_values = preservation_review["raw_item"].map(
        lambda item: raw_field_as_text(item, raw_field)
    ).astype("string")
    parsed_values = preservation_review[parsed_field].astype("string")
    preservation_review[f"{parsed_field}_matches"] = (
        raw_values.fillna("<missing>").eq(parsed_values.fillna("<missing>"))
    )

comparison_fields = ["text_matches", "source_key_matches", "level_matches", "page_matches"]
print("Items present on both sides:", preservation_review["_merge"].eq("both").sum())
display((~preservation_review[comparison_fields]).sum().astype(int).rename("entry_disagreements").to_frame())
display(preservation_review.loc[~preservation_review[comparison_fields].all(axis=1)].head(20))

Objective: Inspect raw properties with no dedicated parsed field. A `label` can supply parsed text or carry additional numbering; compare before deciding it is redundant. Presence does not establish that every value is useful.

In [ ]:
additional_properties = raw_properties.loc[
    raw_properties["property"].isin(["label", "subtitle", "authors", "description", "class"])
]
display(additional_properties.groupby("property").agg(
    items_with_key=("raw_position", "size"),
    books_with_key=("parent_asin", "nunique"),
))
with pd.option_context("display.max_colwidth", 140):
    display(additional_properties.groupby("property", sort=True).head(5))

Objective: Measure missing and blank values separately for each parsed field. Percentages use all parsed entries as the denominator.

In [ ]:
coverage_rows = []
for field in ["text", "sequence", "level", "page", "source_key"]:
    values = parsed_items[field].astype("string")
    coverage_rows.append({
        "field": field,
        "missing_entries": int(values.isna().sum()),
        "blank_entries": int(values.str.strip().eq("").sum()),
        "missing_percent": round(100 * values.isna().mean(), 2),
    })
display(pd.DataFrame(coverage_rows).set_index("field"))

## 6. Inspect order, hierarchy, and pages

Objective: Check sequence numbering, observe level values, and identify books with multiple levels. A flat or missing level does not establish that chapters have no hierarchy.

In [ ]:
sequence_values = pd.to_numeric(parsed_items["sequence"], errors="coerce")
sequence_disagreements = sequence_values.ne(parsed_items["parsed_position"])
print("Sequence differs from actual position:", int(sequence_disagreements.sum()))
display(parsed_items["level"].value_counts(dropna=False).rename("entries").to_frame())

level_values = pd.to_numeric(parsed_items["level"], errors="coerce")
level_steps = level_values.groupby(parsed_items["parent_asin"]).diff()
large_level_jumps = level_steps.gt(1)
print("Entries with a downward hierarchy jump larger than one level:", int(large_level_jumps.sum()))
display(parsed_items.loc[large_level_jumps].head(10))

levels_per_book = parsed_items.groupby("parent_asin")["level"].nunique()
print("Books with multiple nonmissing levels:", int(levels_per_book.gt(1).sum()))
print("Books without any level:", int(levels_per_book.eq(0).sum()))

Objective: Inspect page values that are not simple integers, such as Roman numerals or ranges. These are review cases, not automatically invalid pages.

In [ ]:
page_values = parsed_items["page"].astype("string")
present_pages = page_values.notna() & page_values.str.strip().ne("")
integer_pages = page_values.str.fullmatch(r"[0-9]+").fillna(False)
other_pages = present_pages & ~integer_pages
print("Entries with a page:", int(present_pages.sum()))
print("Entries with a noninteger page representation:", int(other_pages.sum()))
display(page_values.loc[other_pages].value_counts().head(20).rename("entries").to_frame())

## 7. Inspect text quality and repetition

Objective: Flag text needing inspection, including blank text, markup, control characters, and numeric-only labels. Flags can overlap and do not trigger cleaning.

In [ ]:
entry_text = parsed_items["text"].astype("string")
text_flags = pd.DataFrame({
    "missing_text": entry_text.isna(),
    "blank_text": entry_text.str.strip().eq(""),
    "surrounding_whitespace": entry_text.ne(entry_text.str.strip()),
    "html_like_markup": entry_text.str.contains(r"<[^>]+>", regex=True),
    "replacement_character": entry_text.str.contains("\ufffd", regex=False),
    "control_characters": entry_text.str.contains(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", regex=True),
    "numeric_only": entry_text.str.fullmatch(r"\s*[0-9]+\s*"),
    "line_breaks": entry_text.str.contains(r"[\r\n]", regex=True),
}).fillna(False)
flag_rows = []
for flag in text_flags.columns:
    flagged = text_flags[flag]
    flag_rows.append({
        "check": flag,
        "entries": int(flagged.sum()),
        "books": parsed_items.loc[flagged, "parent_asin"].nunique(),
    })
display(pd.DataFrame(flag_rows).set_index("check"))
display(parsed_items.loc[text_flags.any(axis=1), ["parent_asin", "parsed_position", "text"]].head(20))

Objective: Find repeated text within the same book. The second check ignores case and repeated whitespace only for comparison. Repeated headings can be legitimate, so no entries are removed.

In [ ]:
duplicate_review = parsed_items[["parent_asin", "parsed_position", "text"]].copy()
duplicate_review["comparison_text"] = entry_text.str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
exact_repeats = duplicate_review.duplicated(["parent_asin", "text"], keep=False)
normalized_repeats = duplicate_review.duplicated(["parent_asin", "comparison_text"], keep=False)
for name, repeated in [("Exact repeated text", exact_repeats), ("Case/whitespace-normalized repeated text", normalized_repeats)]:
    print(f"{name}: {repeated.sum():,} entries in {duplicate_review.loc[repeated, 'parent_asin'].nunique():,} books")
display(duplicate_review.loc[normalized_repeats].head(20))

Objective: Measure entry and per-book text lengths to understand future chunking needs. Whitespace word counts are descriptive and are not model token counts.

In [ ]:
text_lengths = parsed_items[["parent_asin"]].copy()
text_lengths["characters"] = entry_text.str.len()
text_lengths["whitespace_words"] = entry_text.str.split().str.len()
book_text_lengths = text_lengths.groupby("parent_asin")[["characters", "whitespace_words"]].sum()
display(text_lengths[["characters", "whitespace_words"]].describe(percentiles=[0.5, 0.9, 0.99]).T)
display(book_text_lengths.describe(percentiles=[0.5, 0.9, 0.99]).T)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
toc_counts["actual_parsed_count"].hist(bins=40, ax=axes[0])
axes[0].set(xlabel="Parsed entries per book", ylabel="Books")
text_lengths["characters"].hist(bins=40, ax=axes[1])
axes[1].set(xlabel="Characters per entry", ylabel="Entries")
plt.tight_layout()
plt.show()
display(books.loc[books["parent_asin"].isin(book_text_lengths.nlargest(5, "characters").index),
                  ["parent_asin", "title", "toc_entry_count"]])

## 8. Review what the TOC describes

Objective: Find examples mentioning volumes, parts, chapters, or common front/back matter. These English patterns are incomplete inspection aids; they do not classify the whole TOC or decide which entries to keep.

In [ ]:
content_patterns = {
    "volume_marker": r"^\s*(?:v\.|vol\.?|volume)\s+[0-9ivxlcdm]+\b",
    "part_marker": r"^\s*part\s+[0-9ivxlcdm]+\b",
    "chapter_marker": r"^\s*(?:chapter|ch\.)\s+[0-9ivxlcdm]+\b",
    "front_or_back_matter": r"^\s*(?:introduction|preface|foreword|acknowledg\w*|bibliography|index|appendix|appendices)\b",
}
for label, pattern in content_patterns.items():
    matches = entry_text.str.contains(pattern, case=False, regex=True, na=False)
    print(f"{label}: {matches.sum():,} entries; {parsed_items.loc[matches, 'parent_asin'].nunique():,} books")
    display(parsed_items.loc[matches, ["parent_asin", "parsed_position", "text"]].head(5))

Objective: Search other saved text for explicit contents headings. Matches are candidates for manual review and may just advertise a TOC; absence of a match does not rule out chapter information. Nested property names are searched separately to locate structured TOCs.

In [ ]:
def contents_key_paths(value, path=""):
    paths = set()
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}" if path else key
            if re.search(r"toc|contents?|chapters?|outline", key, re.IGNORECASE):
                paths.add(child_path)
            paths.update(contents_key_paths(child, child_path))
    elif isinstance(value, list):
        for child in value:
            paths.update(contents_key_paths(child, path + "[]"))
    return paths


key_path_rows = []
for column in ["amazon_raw_record_json", "openlibrary_raw_edition_json"]:
    for parent_asin, record in zip(books["parent_asin"], books[column].map(json.loads), strict=True):
        for path in contents_key_paths(record):
            key_path_rows.append({"source": column, "path": path, "parent_asin": parent_asin})
key_path_summary = pd.DataFrame(key_path_rows, columns=["source", "path", "parent_asin"])
display(key_path_summary.groupby(["source", "path"]).size().rename("books").to_frame())

Objective: Display possible contents references in Amazon descriptions/features and Open Library descriptions/notes. This phrase search is deliberately limited to explicit headings; inspect the full values before interpreting them as TOC content.

In [ ]:
contents_heading_pattern = r"\b(?:table\s+of\s+contents|list\s+of\s+chapters|contents\s*:)"
text_candidates = []
for position, book in enumerate(books.itertuples()):
    edition = openlibrary_records.iloc[position]
    fields = {
        "amazon_description": list(book.description),
        "amazon_features": list(book.features),
        "openlibrary_description": edition.get("description"),
        "openlibrary_notes": edition.get("notes"),
    }
    for field, value in fields.items():
        text = json.dumps(value, ensure_ascii=False)
        match = re.search(contents_heading_pattern, text, re.IGNORECASE)
        if match:
            text_candidates.append({
                "row_position": position,
                "parent_asin": book.parent_asin,
                "field": field,
                "excerpt": text[max(0, match.start() - 60):match.end() + 240],
            })
hidden_text_review = pd.DataFrame(text_candidates, columns=["row_position", "parent_asin", "field", "excerpt"])
display(hidden_text_review.groupby("field").size().rename("candidate_books").to_frame())
with pd.option_context("display.max_colwidth", None):
    display(hidden_text_review.head(10))

## 9. Confirm the raw file is unchanged

Objective: Compare the before and after checksums. Keep all cleaning, removal, imputation, feature-selection, and chunking decisions for a later phase with an agreed evaluation design.

In [ ]:
checksum_after = file_checksum(raw_path)
print(f"Checksum before: {checksum_before}")
print(f"Checksum after:  {checksum_after}")
assert checksum_after == checksum_before, "The raw Parquet changed during exploration."
print("Raw file unchanged.")

### Questions to resolve through manual inspection

- Does the TOC describe the matched edition, a multivolume work, or a subset of the book?
- Which raw labels, subtitles, author credits, or descriptions add meaning beyond parsed text?
- Do repeated headings reflect actual sections, duplicated source data, or parsing issues?
- What do absent levels/pages mean for each source format?
- Are phrase-search candidates actual chapter lists or mentions of a TOC?

No automatic correction follows from these flags. This catalog has been explored as development evidence. Define retrieval evaluation before selecting preprocessing or chunking strategies. Record findings in [docs/data.md](../docs/data.md).